<a href="https://colab.research.google.com/github/saatour/GW170817/blob/main/20261013_EAGLE_simulation_analysis_Chapter6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%pip install numpy pandas matplotlib scipy astropy healpy requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 26.7 MB/s eta 0:00:00


In [2]:
from pathlib import Path
import hashlib
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import healpy as hp
import requests

from astropy.io import fits
from scipy.integrate import cumulative_trapezoid

# Create local data directory
DATA = Path("data")
DATA.mkdir(exist_ok=True)

# GW190814 localisation file
SKYMAP_URL = (
    "https://dcc.ligo.org/LIGO-P2000230/public/"
    "GW190814_skymap.fits.gz"
)

SKYMAP_FILE = DATA / "GW190814_skymap.fits.gz"

# Credible probability level
CREDIBLE = 0.90

print("Working directory:", Path.cwd().resolve())
print("Target credible level:", CREDIBLE)

Working directory: /content
Target credible level: 0.9


In [3]:
if not SKYMAP_FILE.exists():

    response = requests.get(
        SKYMAP_URL,
        timeout=120
    )

    response.raise_for_status()

    SKYMAP_FILE.write_bytes(response.content)

# Confirm file exists
assert SKYMAP_FILE.exists()
assert SKYMAP_FILE.stat().st_size > 1000

# Record a checksum for reproducibility
sha256 = hashlib.sha256(
    SKYMAP_FILE.read_bytes()
).hexdigest()

print("File:", SKYMAP_FILE)
print("File size:", SKYMAP_FILE.stat().st_size, "bytes")
print("SHA256:", sha256)

# Inspect the FITS structure
with fits.open(SKYMAP_FILE) as hdul:

    hdul.info()

    print("\nAvailable columns:")
    print(hdul[1].columns.names)

    print("\nHEALPix ordering:")
    print(hdul[1].header.get("ORDERING"))

File: data/GW190814_skymap.fits.gz
File size: 1472321 bytes
SHA256: 468ec8032ed11458ff9ef3c8735a039fcb0ead37f7a57be9c133f3df2dd8757e
Filename: data/GW190814_skymap.fits.gz
No.    Name      Ver    Type      Cards   Dimensions   Format
  0  PRIMARY       1 PrimaryHDU       4   ()      
  1                1 BinTableHDU     32   12582912R x 4C   [D, D, D, D]   

Available columns:
['PROB', 'DISTMU', 'DISTSIGMA', 'DISTNORM']

HEALPix ordering:
NESTED


In [4]:
with fits.open(SKYMAP_FILE, memmap=False) as hdul:

    header = hdul[1].header
    table = hdul[1].data

    required_columns = {
        "PROB",
        "DISTMU",
        "DISTSIGMA",
        "DISTNORM"
    }

    available_columns = set(table.names)

    assert required_columns.issubset(available_columns), (
        f"Missing columns: "
        f"{required_columns - available_columns}"
    )

    nside = int(header["NSIDE"])
    ordering = str(header.get("ORDERING", "")).upper()

    prob = np.asarray(table["PROB"], dtype=float)
    distmu = np.asarray(table["DISTMU"], dtype=float)
    distsigma = np.asarray(table["DISTSIGMA"], dtype=float)
    distnorm = np.asarray(table["DISTNORM"], dtype=float)

# Validate prob map
assert len(prob) == hp.nside2npix(nside)
assert np.all(np.isfinite(prob))
assert np.all(prob >= 0)
assert np.isclose(prob.sum(), 1.0, atol=1e-3)

print("NSIDE:", nside)
print("Ordering:", ordering)
print("Number of pixels:", len(prob))
print("Total probability:", prob.sum())

NSIDE: 1024
Ordering: NESTED
Number of pixels: 12582912
Total probability: 0.9999999999999997


In [8]:
from astropy.cosmology import Planck18, z_at_value
import astropy.units as u

# We are calculating a true 3D credible region,
# not combining separate sky and distance intervals.

CREDIBLE = 0.90

# HEALPix pixel solid angle
pixel_area_sr = hp.nside2pixarea(nside)

# Pixels with valid conditional distance information
valid = (
    np.isfinite(prob)
    & (prob > 0)
    & np.isfinite(distmu)
    & np.isfinite(distsigma)
    & np.isfinite(distnorm)
    & (distsigma > 0)
    & (distnorm > 0)
)

print("Probability with valid distance information:",
      prob[valid].sum())

assert prob[valid].sum() > 0.999, (
    "Too much probability is missing distance information."
)

# Set a luminosity-distance range covering the posterior
distance_max = np.max(
    distmu[valid] + 8 * distsigma[valid]
)

# Radial discretisation
N_RADIAL = 1200

dl_edges = np.linspace(
    0.0,
    distance_max,
    N_RADIAL + 1
)

dl_mid = 0.5 * (dl_edges[:-1] + dl_edges[1:])
dl_width = np.diff(dl_edges)

# Convert luminosity-distance edges to comoving distance.
# We use Planck18 as the reference cosmology.

z_max = z_at_value(
    Planck18.luminosity_distance,
    distance_max * u.Mpc,
    zmax=10
)

z_lookup = np.linspace(
    0,
    float(z_max) * 1.01,
    5000
)

dl_lookup = Planck18.luminosity_distance(
    z_lookup
).to_value(u.Mpc)

dc_lookup = Planck18.comoving_distance(
    z_lookup
).to_value(u.Mpc)

dc_edges = np.interp(
    dl_edges,
    dl_lookup,
    dc_lookup
)

# Exact comoving shell volume per steradian
shell_volume = (
    dc_edges[1:]**3 - dc_edges[:-1]**3
) / 3.0

# Volume of one HEALPix angular pixel
# intersected with each radial shell
voxel_volume = pixel_area_sr * shell_volume

print(f"HEALPix NSIDE: {nside}")
print(f"Valid sky pixels: {valid.sum():,}")
print(f"Maximum luminosity distance: {distance_max:.2f} Mpc")
print(f"Radial shells: {N_RADIAL}")
print(f"Maximum comoving distance: {dc_edges[-1]:.2f} Mpc")

Probability with valid distance information: 0.9999999999999999
HEALPix NSIDE: 1024
Valid sky pixels: 171,136
Maximum luminosity distance: 708.56 Mpc
Radial shells: 1200
Maximum comoving distance: 618.99 Mpc


In [9]:
# Extract valid pixel data
p = prob[valid]
mu = distmu[valid]
sigma = distsigma[valid]
norm = distnorm[valid]

# Probability density per comoving Mpc^3 is used
# to define the highest-posterior-density region.

# Logarithmic bins for numerical density thresholding
density_edges = np.logspace(-40, 0, 6001)
n_bins = len(density_edges) - 1

prob_hist = np.zeros(n_bins)
volume_hist = np.zeros(n_bins)

total_probability = 0.0
outside_density_range = 0.0

CHUNK_SIZE = 128

for start in range(0, len(p), CHUNK_SIZE):

    end = min(start + CHUNK_SIZE, len(p))

    pc = p[start:end, None]
    muc = mu[start:end, None]
    sigc = sigma[start:end, None]
    normc = norm[start:end, None]

    # Conditional luminosity-distance posterior:
    # p(D_L | sky) proportional to
    # D_L^2 exp[-(D_L - mu)^2 / (2 sigma^2)]

    exponent = -0.5 * (
        (dl_mid[None, :] - muc) / sigc
    )**2

    conditional_pdf = (
        normc
        * dl_mid[None, :]**2
        * np.exp(exponent)
        / (np.sqrt(2 * np.pi) * sigc)
    )

    # Joint probability in each sky/radial voxel
    voxel_prob = (
        pc
        * conditional_pdf
        * dl_width[None, :]
    )

    total_probability += voxel_prob.sum()

    # Probability density per unit comoving volume
    density = voxel_prob / voxel_volume[None, :]

    # Assign each voxel to a density histogram bin
    indices = np.searchsorted(
        density_edges,
        density,
        side="right"
    ) - 1

    in_range = (
        (indices >= 0)
        & (indices < n_bins)
        & np.isfinite(density)
    )

    outside_density_range += voxel_prob[~in_range].sum()

    # Histogram of posterior probability
    prob_hist += np.bincount(
        indices[in_range],
        weights=voxel_prob[in_range],
        minlength=n_bins
    )

    # Histogram of geometric comoving volume
    volumes = np.broadcast_to(
        voxel_volume[None, :],
        voxel_prob.shape
    )

    volume_hist += np.bincount(
        indices[in_range],
        weights=volumes[in_range],
        minlength=n_bins
    )

print(f"Integrated posterior probability: {total_probability:.6f}")
print(f"Probability outside density bins: {outside_density_range:.3e}")

assert np.isclose(
    total_probability, 1.0, atol=0.005
), "Posterior integration is not sufficiently normalised."

assert outside_density_range < 1e-6, (
    "Density histogram bounds are insufficient."
)

Integrated posterior probability: 1.000000
Probability outside density bins: 1.595e-36


In [10]:
# Numerical probability normalisation
prob_hist_normalised = prob_hist / total_probability

# Reverse arrays: highest density first
p_desc = prob_hist_normalised[::-1]
v_desc = volume_hist[::-1]

cumulative_probability = np.cumsum(p_desc)
cumulative_volume = np.cumsum(v_desc)

# First bin reaching the requested probability
idx = np.searchsorted(
    cumulative_probability,
    CREDIBLE
)

assert idx < len(p_desc), (
    "Could not enclose the requested probability."
)

# Probability enclosed before the boundary bin
p_before = (
    cumulative_probability[idx - 1]
    if idx > 0 else 0.0
)

# Volume enclosed before the boundary bin
v_before = (
    cumulative_volume[idx - 1]
    if idx > 0 else 0.0
)

# Fraction of the boundary density bin needed
fraction = (
    (CREDIBLE - p_before) / p_desc[idx]
)

# Approximate HPD volume,
# interpolating inside the boundary density bin
V90 = v_before + fraction * v_desc[idx]

# Equivalent cubic volume
L_cube = V90**(1.0 / 3.0)

print("=" * 55)
print("GW190814 THREE-DIMENSIONAL LOCALISATION")
print("=" * 55)

print(f"Credible probability: {CREDIBLE:.1%}")

print(
    f"90% HPD comoving volume: "
    f"{V90:,.2f} cMpc^3"
)

print(
    f"Equivalent EAGLE cube side: "
    f"{L_cube:.3f} cMpc"
)

print(
    f"Posterior probability represented: "
    f"{total_probability:.6f}"
)

print("=" * 55)

# Save results for subsequent notebook cells
volume_results = {
    "event": "GW190814",
    "credible_level": CREDIBLE,
    "definition": "3D highest-posterior-density region",
    "cosmology": "Planck18",
    "volume_comoving_mpc3": float(V90),
    "cube_side_comoving_mpc": float(L_cube),
    "radial_shells": N_RADIAL,
    "integrated_probability": float(total_probability),
    "skymap_sha256": sha256
}

output_file = DATA / "GW190814_volume_results.json"

with open(output_file, "w") as f:
    json.dump(volume_results, f, indent=2)

print(f"Results saved to: {output_file}")

GW190814 THREE-DIMENSIONAL LOCALISATION
Credible probability: 90.0%
90% HPD comoving volume: 31,666.01 cMpc^3
Equivalent EAGLE cube side: 31.637 cMpc
Posterior probability represented: 1.000000
Results saved to: data/GW190814_volume_results.json
